In [1]:
include("CRD_STA.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using ProgressMeter
using DelimitedFiles
using PyCall

In [ ]:
Tw = 0.9
N_cheb = 149
Mr = 0.3
gamma = 1.4
sigma = 0.72
omega = 0.024
R = 60
c = 0.2
Ma = Mr/R
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,1,0);
H,T = T_ca(Mr,f,q,w0,gamma,Tw)
F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim");
lam = - (2/3) * T
kappa = (1/sigma) * T
num = 1
be = 0.
A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,1,0)
nep = PEP([A0,A1,A2]); 
eigval,eigvec = iar(nep , σ = c , neigs = num ,maxit = 500,tol=1e-10)
eigval = conj(eigval)

/home/zhj/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integr

1-element Vector{ComplexF64}:
 0.6044401792826707 + 0.00034297686702940296im

In [ ]:
for Tw = 0.8 : 0.1 : 1.2
        N_cheb = 149
        Mr = 0.3
        gamma = 1.4
        sigma = 0.72
        be_start = -0.015
        be_end = 0.22
        be_step = 0.001
        R_end = 60
        R_step = 0.1
        Ro = 1
        Co = 0
        global u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
        H,T = T_ca(Mr,f,q,w0,gamma,Tw)
        F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
        lam = - (2/3) * T
        kappa = (1/sigma) * T
    for omega = -0.11 
        global GR = 0
            for  R = 24.8 : 0.2 : 30

                global initial_i_2 = []
                global initial_r_2 = []
                global tempvec_i = [0]
                writedlm("output_eig.dat",initial_r_1)
                writedlm("output_GR_$omega _$Tw.dat",initial_r_1)
                writedlm("output_GR_data_$omega _$Tw.dat",initial_r_1)
                for be = be_start : 0.25*be_step : be_end
                    Ma = Mr/R
                    A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co)
                    nep = PEP([A0,A1,A2]); 
                    eigval,eigvec = iar(nep,σ = 0.1 , neigs = 1 ,maxit = 500,tol=1e-10)
                    eigval = conj(eigval)
                    global tempvec_i = [tempvec_i;imag(eigval[1])]
                    global tempvec_r = [tempvec_i;real(eigval[1])]

                    open("output_eig.dat", "a") do io
                        write(io,"R = $R,be = $be,eig = $eigval\n")
                    end
                    if imag(eigval[1]) * tempvec_i[end - 1,1] < 0 && abs.(imag(eigval[1]))<0.03
                        global initial_i_2 = [omega R be imag(eigval[1])]
                        global initial_r_2 = [omega R be real(eigval[1])]
                        global total_r_2 = initial_r_2
                        global total_i_2 = initial_i_2
                        global mode = 1
                        break
                    elseif length(tempvec_i) > 3  && tempvec_i[end - 1,1] < tempvec_i[end - 2,1] && tempvec_i[end - 1,1] < imag(eigval[1])
                        global mode = 0
                        break
                    end
                end
                if  mode == 1
                    global R_start = initial_i_2[end,2]
                    global beta = initial_i_2[end,3]
                    break
                end
            end
        for R = R_start  : R_step : R_end
            Ma = Mr/R
            A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,beta,N_cheb,1,0)
            nep = PEP([A0,A1,A2]); 

            eigval_2,eigvec = iar(nep , σ = total_r_2[end,4] - im * total_i_2[end,4] , neigs = 1 ,maxit = 500,tol=1e-10)
            eigval_2 = conj(eigval_2)
            GR_temp = imag(eigval_2[1]) * R_step
            global GR = GR + abs(GR_temp)
            open("output_GR_$omega _$Tw.dat", "a") do io
            write(io,"R = $R , eigmode2 = $eigval_2 , GR = $GR\n")
            end

            open("output_GR_data_$omega _$Tw.dat", "a") do io
            write(io,"$beta\t$R\t$GR\n")
            end

            if imag(eigval_2[1]) < 0
                total_i_2 = [omega R beta imag(eigval_2[1])]
                total_r_2 = [omega R beta real(eigval_2[1])]
            end
        end
    end
end

InterruptException: InterruptException: